# 01 · Record an orobot teleoperation episode

Connect to an orobot robot from a notebook, then record a teleoperation session
(joint states over time) to a local on-disk episode.

While recording, **drive the robot from the orobot.io browser UI** — the session
captures the resulting joint stream. Press the stop button (or Ctrl-C in a local
kernel) to end the episode.

In [ ]:
%pip install orobot  # zero third-party deps; installs instantly

## Authenticate

Mint an API key in the orobot developer portal (`POST /api/user/api-keys`) with the
`robots:read` scope. In Colab, store it as a secret rather than pasting it inline.

In [ ]:
import os
from orobot import OrobotClient

API_KEY = os.environ.get('OROBOT_API_KEY', 'ork_live_...')
client = OrobotClient(api_key=API_KEY)

robots = client.list_robots()
for r in robots:
    print(r.get('uuid'), '-', r.get('name'))

## Pick a robot and inspect its current joint state

In [ ]:
robot_uuid = robots[0]['uuid']
print('current joints:', client.get_joint_state(robot_uuid))

## Record

The block below opens an episode and blocks until you stop it. The live joint
stream over the `/control` WebSocket is wired by the platform-side recording
pipeline (orobotio#3252); until that lands you can drive the recorder manually by
polling and calling `session.add_frame(...)`, shown in the fallback cell.

In [ ]:
with client.record(robot_uuid, output_dir='./data/episode_0', fps=30) as session:
    session.wait_for_stop()  # teleop in the browser; Ctrl-C / stop to finish

print('saved', len(session.episode.frames), 'frames to', session.episode.output_dir)

### Fallback: manual polling loop (works today)

Until the live WS stream lands, poll the robot state and append frames yourself.

In [ ]:
import time

with client.record(robot_uuid, output_dir='./data/episode_0_polled', fps=10) as session:
    for _ in range(50):  # ~5s at 10 Hz
        session.add_frame(client.get_joint_state(robot_uuid))
        time.sleep(0.1)
print('saved', len(session.episode.frames), 'frames')

## Inspect the saved episode

```
./data/episode_0/
  meta.json      # robot uuid, fps, joint names, frame count
  frames.jsonl   # one {t, joints, images, action} per line
```

In [ ]:
import json, pathlib
meta = json.loads(pathlib.Path('./data/episode_0_polled/meta.json').read_text())
print(meta)

---
Next: [`02_export_to_hf_hub.ipynb`](02_export_to_hf_hub.ipynb) converts these
episodes to a Hugging Face `LeRobotDataset`.